# Visualize results

This notebook allows the analysis and visualization of a benchmark run

## Imports

In [ ]:
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

## Some variables

In [ ]:
pd.options.plotting.backend = "plotly"

results_files = [
    "", "", "" # enter results files to analyse
]

image_dir = "."
create_images = False

custom_color_map = {
    'general QA': '#00274A',
    'multiple choice': '#EC9A29',
    'fact checking': '#A8201A',
    'multi-hop QA': '#6C698D',
    'corporate QA': '#0EA4E3'
}

metrics = [
    "context_precision","context_recall","nv_context_relevance","answer_relevancy",
    "nv_response_groundedness", "correctness","semantic_similarity",
    "trustworthiness","prompt_alignment"
]

## Load Data

In [ ]:
flat_results = []

for run_file in results_files:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                flat_results.append(r)

## Transform and clean

In [ ]:
df = pd.DataFrame(flat_results)

# clean the data if necessary

## Calculate results

Calculate the results of the run on different abstraction layers to allow overall comparison and in-detail analysis

In [ ]:
overall_result = df[metrics].mean().mean()

detail = df.groupby('dataset')[metrics].mean()

results_by_dataset = df.groupby('dataset')[metrics].mean().T.mean()

results_by_metric = df[metrics].mean()

## Visualize Results

In [ ]:


fig = go.Figure(go.Indicator(
    mode = "number",
    number={'prefix': 'Overall Score: '},
    value = overall_result))

fig.show()

if create_images:
    pio.write_image(fig, f'{image_dir}/overall-score.pdf')

In [ ]:
display(results_by_metric)

pd.DataFrame(results_by_dataset).plot.bar(
    color=results_by_dataset.index, 
    color_discrete_map=custom_color_map,
    labels={"dataset": "dataset", "value": "mean score"},
    range_y=[0, 1]
)

ds_means_fig = go.Figure()

for dataset in results_by_dataset.index:
    y = results_by_dataset[dataset]
    color = "#FF7B52" if y <= 0.5 else "#FFC44F" if (y <= 0.7 and y > 0.5) else "#F1FF53" if (y <= 0.9 and y > 0.7) else "#56D752"
    ds_means_fig.add_trace(
        go.Bar(x=[dataset], y=[results_by_dataset[dataset]], marker_color=color, text=np.round(y, 2))
    )

ds_means_fig.update_layout(
    xaxis_title='Dataset',
    yaxis_title='Mean score',
    showlegend=False,
    height=500,
    width=450,
    font=dict(
        size=16,
    ),
)

ds_means_fig.show()

if create_images:
    pio.write_image(ds_means_fig, f'{image_dir}/dataset-means-bar.pdf')

In [ ]:
metrics = results_by_dataset.index.tolist()
values = results_by_dataset.values.tolist()

fig = px.line_polar(
    r=values,
    theta=metrics,
    line_close=True,
    color_discrete_sequence=["#00274A"]
)

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True, 
            range=[0, 1],
        )
    ),
    font=dict(
        size=16,
    ),
)

fig.show()

In [ ]:
results_by_metric.rename("mean score", inplace=True)

display(results_by_metric)

mt_means_fig = go.Figure()

for metric in results_by_metric.index:
    y = results_by_metric[metric]
    color = "#FF7B52" if y <= 0.5 else "#FFC44F" if (y <= 0.7 and y > 0.5) else "#F1FF53" if (y <= 0.9 and y > 0.7) else "#56D752"
    mt_means_fig.add_trace(
        go.Bar(x=[metric], y=[results_by_metric[metric]], marker_color=color, text=np.round(y, 2))
    )

mt_means_fig.update_layout(
    xaxis_title='Metric',
    yaxis_title='Mean score',
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    ),
)

mt_means_fig.show()

if create_images:
    pio.write_image(mt_means_fig, f'{image_dir}/metric-means-bar.pdf')

In [ ]:
metrics = results_by_metric.index.tolist()
values = results_by_metric.values.tolist()

fig = px.line_polar(
    r=values,
    theta=metrics,
    line_close=True,
    color_discrete_sequence=["#00274A"]
)

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True, 
            range=[0, 1],
        )
    ),
    font=dict(
        size=16,
    ),
)

fig.show()

In [ ]:
import plotly.express as px

text_matrix = detail.round(2).astype(str)
text_matrix = text_matrix.replace("nan", "n.a.")

deatil_fig = go.Figure(data=go.Heatmap(
    z=detail,
    x=detail.columns,
    y=detail.index,
    text=text_matrix,
    zmin=0.0,
    zmax=1.0,
    texttemplate="%{text}",
    colorscale=[(0.00, "#FF7B52"), (0.5, "#FF7B52"), 
                (0.5, "#FFC44F"), (0.7, "#FFC44F"),
                (0.7, "#F1FF53"), (0.9, "#F1FF53"),
                (0.9, "#56D752"),  (1.00, "#56D752")]
))

deatil_fig.update_layout(
    xaxis_title="Metrics",
    yaxis_title="Datasets",
    xaxis_nticks=len(detail.columns),
    yaxis_nticks=len(detail.index),
    font=dict(
        size=16,
    ),

)
    
deatil_fig.show()

if create_images:
    pio.write_image(deatil_fig, f'{image_dir}/detail-scores.png', scale=5,)

## Distribution and Outliers

First clean raw dataframe (see conditions above)

In [ ]:
df.loc[ (df['dataset'] == 'fact checking'), 'context_precision'] = float('nan')
df.loc[ (df['dataset'] == 'fact checking'), 'context_recall'] = float('nan')
df.loc[ (df['dataset'] == 'fact checking'), 'answer_relevancy'] = float('nan')
df.loc[ (df['dataset'] == 'fact checking'), 'semantic_similarity'] = float('nan')
df.loc[ (df['dataset'] == 'fact checking'), 'trustworthiness'] = float('nan')
df.loc[ (df['dataset'] == 'fact checking'), 'nv_response_groundedness'] = float('nan')

df.loc[ (df['dataset'] == 'multiple choice'), 'answer_relevancy'] = float('nan')

In [ ]:
df_melt = df.melt(
    id_vars=['dataset'],
    value_vars=metrics,
    var_name='Metric',
    value_name='Value'
)

fig = px.box(
    df_melt, 
    x='Metric',
    y='Value',
    color='dataset',
    facet_col='Metric',
    facet_col_wrap=4,
    color_discrete_map=custom_color_map,
    height=600
)

fig.update_xaxes(matches=None, showticklabels=False, title_text="")
fig.update_yaxes(matches=None)

fig.show()

In [ ]:

for metric in metrics:
    flat_df = pd.DataFrame(flat_results)
    for dataset in flat_df['dataset'].unique():
        result_df = flat_df.query('dataset == @dataset')
        upper_quartile = result_df[metric].quantile(0.75)
        lower_quartile = result_df[metric].quantile(0.25)
        iqr = upper_quartile - lower_quartile
        threshold = lower_quartile - 1.5 * iqr
        outlier = result_df[result_df[metric] < threshold]
        if not outlier.empty:
            print(f"{dataset} outlier for {metric} (worse than {threshold}):")
            display(outlier)